# Phase 4: Field Plots & Ground Truth

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedRafiAhmedGIS/forest-carbon-pipeline/blob/main/notebooks/04_Field_Plots_Ground_Truth.ipynb)

Precise trigonometry anchoring ground truth on 20x20m plots. Enforces Indian FSI 30 cm CBH threshold.


In [ ]:
# Setup: locate the repo's `src` package (any cwd inside the repo, or fresh clone)
import os
import subprocess
import sys

REPO = "forest-carbon-pipeline"
if os.path.isdir("src"):
    ROOT = os.getcwd()
elif os.path.isdir(os.path.join("..", "src")):
    ROOT = os.path.abspath("..")
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/SyedRafiAhmedGIS/forest-carbon-pipeline.git"],
                   check=False)
    ROOT = os.path.abspath(REPO)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
if not os.path.isdir(os.path.join(ROOT, "src")):
    raise RuntimeError(
        f"Could not locate the pipeline source at {ROOT}. Open this notebook "
        "from a repo checkout, or check your network so the bootstrap can "
        "clone https://github.com/SyedRafiAhmedGIS/forest-carbon-pipeline"
    )

from src.phase1_demarcate import demarcate_forest_pixels  # noqa: E402
QUARANTINE = os.path.join(ROOT, "quarantine")
print(f"Pipeline package ready (ROOT = {ROOT})")


## Synthetic scene

Regenerated deterministically so this notebook runs standalone.


In [ ]:
# Synthetic Sentinel-2-like scene: 100 x 100 px at 20 m (deterministic)
import numpy as np

np.random.seed(42)
nir_band = np.random.uniform(0.2, 0.8, (100, 100))
red_band = np.random.uniform(0.05, 0.3, (100, 100))
savi = ((nir_band - red_band) / (nir_band + red_band + 0.5)) * 1.5  # L = 0.5
print("Scene ready: 100x100 px @ 20 m, seed=42")

forest_mask, ndvi, input_hash = demarcate_forest_pixels(
    nir_band, red_band, quarantine_dir=QUARANTINE)


## Indian FSI practice (30 cm CBH)

Only stems with CBH >= 30 cm count as mature trees; stems of 10-30 cm are saplings and are logged but excluded from AGB. Each plot is 20 x 20 m (0.04 ha), trees are measured with a laser hypsometer and calliper, and a plot qualifies for modelling when it holds at least 3 mature stems.


In [ ]:
from src.phase2_preprocess import classify_spectral_types
from src.phase3_clubbing import clubbing_stratification
from src.phase4_model import generate_field_campaign

type_map, density_map = classify_spectral_types(ndvi, savi, forest_mask)
stratum_map, allocation = clubbing_stratification(type_map, density_map)
plots = generate_field_campaign(ndvi, savi, forest_mask, stratum_map)
print("Sample plot record:", plots[0])
